# Part 3 - 03 YOLOv8 training

Fine-tune a **COCO pre-trained** YOLOv8 on the clean dataset built in `part3_02` (`data/yolo/data.yaml`).

**Run this notebook on the Vast.ai GPU instance.** Preparing it:
```bash
git clone <your-repo-url> && cd NeuroVision
uv sync && uv add ultralytics          # once, if not already in pyproject.toml
# get the data: download the Kaggle dataset into data/raw_bbox, then run part3_01 and part3_02
```
`data/` is not in Git, so the dataset is rebuilt on the instance by notebook 02 (same seed, same result).

Steps: environment check, model choice, configuration, 1-epoch sanity run, full training with timing, outputs, loss curves and overfitting, resuming.

In [ ]:
import json
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

from neurovision.yolo_data import SEED, set_seed
from neurovision.yolo_train import (AUGMENTATION, TRAIN_DEFAULTS, check_environment, load_history,
                                    overfitting_report, save_run_summary, train_model)

set_seed(SEED)

DATA_YAML = Path("../data/yolo/data.yaml").resolve()
RUNS = Path("../runs/detect").resolve()
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)


def show_images(paths, cols=2, figsize=(14, 7)):
    paths = [p for p in paths if Path(p).exists()]
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, p in zip(axes.ravel(), paths):
        ax.imshow(mpimg.imread(p))
        ax.set_title(Path(p).name, fontsize=9)
    plt.tight_layout()
    plt.show()

## 1. Environment and dataset check

In [ ]:
env = check_environment()
print(json.dumps(env, indent=2))
if not env["cuda_available"]:
    print("\nWARNING: no GPU detected. Training on CPU is possible for the 1-epoch test but far too slow for the full run.")

assert DATA_YAML.exists(), "data.yaml not found: run part3_02_data_preprocessing first"
print("\n" + DATA_YAML.read_text())

## 2. Model choice

All three start from COCO weights (transfer learning): the backbone already knows edges, textures and shapes, and only has to adapt to MRI and to 4 classes.

| Model | Parameters | FLOPs (640 px) | COCO mAP50-95 | Use here |
|---|---|---|---|---|
| yolov8n | 3.2 M | 8.7 G | 37.3 | fastest, smallest file; baseline for comparison |
| **yolov8s** | 11.2 M | 28.6 G | 44.9 | **main model**: good accuracy for a 4-class task, trains quickly on one GPU |
| yolov8m | 25.9 M | 78.9 G | 50.2 | slower, more memory; only worth it if `s` underfits |

Reasoning: the task has 4 classes and a few thousand images, so a medium model risks overfitting and costs much more time and memory for little gain. `yolov8s` is the main model; `yolov8n` will be the second MLflow run (one single change: model size).

## 3. Configuration

In [ ]:
MODEL = "yolov8s.pt"

# Image size: use the most common size found in part3_01 (rounded to a multiple of 32).
OVERRIDES = {"imgsz": 640, "batch": 16}   # lower `batch` if you get a CUDA out-of-memory error

print("Model:", MODEL)
print(json.dumps({**TRAIN_DEFAULTS, **OVERRIDES}, indent=2))
print("\nAugmentation:")
print(json.dumps(AUGMENTATION, indent=2))

### Why these hyperparameters
- **epochs=50, patience=10**: 50 is a ceiling; early stopping ends the run when validation fitness stops improving for 10 epochs, which limits overfitting and wasted GPU time.
- **imgsz=640**: YOLOv8's default and a multiple of 32. Check the common image size from notebook 01: if images are 512x512, `imgsz=512` is faster at no loss.
- **batch=16**: limited by GPU memory (reduce if out of memory).
- **AdamW, lr0=0.001**: fine-tuning a pre-trained network needs a small learning rate to keep what it already knows; `lrf=0.01` decays it to 1e-5 by the end.
- **seed=42, deterministic=True**: reproducibility.

### Augmentation choices (training set only: Ultralytics never augments validation or test)
| Setting | Value | Reason |
|---|---|---|
| rotation (`degrees`) | 10 | head position varies slightly between scans; strong rotation would create unrealistic MRIs |
| `scale`, `translate` | 0.3, 0.1 | tumor size and position vary |
| brightness (`hsv_v`) | 0.3 | intensity differs between scanners |
| contrast | no direct argument | covered only by Ultralytics' default Albumentations transforms (if `albumentations` is installed) |
| horizontal flip (`fliplr`) | **0** | the side of the brain carries information; keep anatomy realistic |
| `flipud`, `shear`, `perspective` | 0 | not realistic for MRI |
| hue, saturation | **0** | images are grayscale: there is no color to vary |
| `mosaic` | 0.5 | stitches 4 images: tumors get smaller and appear in unusual surroundings, and anatomy is cut at the seams. It adds variety but can hurt tumor localization, so it is used on half of the batches and switched off for the last 10 epochs (`close_mosaic`) |
| `mixup` | 0 | blending two MRIs creates ghost tumors |

## 4. Sanity run: 1 epoch

Goal: make sure the dataset is read correctly, and measure the time per epoch. In the log, check that the train and validation scans report the expected number of images and **0 corrupt**, and that there are no label warnings.

In [ ]:
sanity = train_model(MODEL, DATA_YAML, name="sanity_1epoch", project=RUNS, epochs=1, **OVERRIDES)

per_epoch = sanity["duration_s"]
print(f"\n1 epoch: {per_epoch:.0f}s. Upper bound for {TRAIN_DEFAULTS['epochs']} epochs: {per_epoch * TRAIN_DEFAULTS['epochs'] / 60:.0f} min "
      "(the first epoch includes start-up time, so the real figure is lower).")

Augmented training samples saved by Ultralytics (check that boxes still sit on the tumors, no flipped brains, grayscale preserved):

In [ ]:
show_images([sanity["save_dir"] / "train_batch0.jpg", sanity["save_dir"] / "train_batch1.jpg"], cols=2, figsize=(14, 7))

## 5. Full training (duration measured with `time`)

In [ ]:
RUN_NAME = f"{Path(MODEL).stem}_full"
run = train_model(MODEL, DATA_YAML, name=RUN_NAME, project=RUNS, **OVERRIDES)

print(f"Training time: {run['duration_s'] / 60:.1f} min")
save_run_summary(run, REPORTS / f"part3_training_{RUN_NAME}.json")

## 6. Where are the results?

In [ ]:
save_dir = run["save_dir"]
print("Run folder:", save_dir, "\n")
for p in sorted(save_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(save_dir))

print("\nbest.pt exists:", run["best"].exists(), "| last.pt exists:", run["last"].exists())

In [ ]:
show_images([save_dir / "results.png", save_dir / "confusion_matrix_normalized.png",
             save_dir / "BoxPR_curve.png", save_dir / "BoxF1_curve.png"], cols=2, figsize=(16, 12))

- `best.pt`: weights of the best epoch (by validation fitness): the model to evaluate and deploy.
- `last.pt`: weights of the last epoch: used to resume an interrupted run.
- `results.csv`: loss and metrics per epoch.

## 7. Loss curves and overfitting

In [ ]:
history = load_history(save_dir)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (kind, (train_col, val_col)) in zip(axes, {"box": ("train/box_loss", "val/box_loss"),
                                                    "class": ("train/cls_loss", "val/cls_loss"),
                                                    "dfl": ("train/dfl_loss", "val/dfl_loss")}.items()):
    ax.plot(history["epoch"], history[train_col], label="train")
    ax.plot(history["epoch"], history[val_col], label="validation")
    ax.set_title(f"{kind} loss")
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(REPORTS / f"part3_loss_curves_{RUN_NAME}.png", dpi=150)
plt.show()

history[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5)

In [ ]:
report = overfitting_report(history)
print(json.dumps(report, indent=2))

**How to read it**
- Training loss going down while the validation loss stagnates or rises = overfitting (what `overfitting_suspected` checks).
- The levers already in place: early stopping (`patience`), augmentation, and a small learning rate. If overfitting is clear, increase augmentation slightly, lower `epochs`, or try `yolov8n`.
- The best epoch for **recall** may differ from the best epoch for mAP; the project asks to favor recall (a missed tumor is the worst error). `best.pt` is chosen by mAP-based fitness, so check recall on the validation curves and in the evaluation notebook before choosing the champion.

## 8. Resuming an interrupted run

In [ ]:
# Only if the run was interrupted (connection lost, instance stopped): continue from last.pt.
RESUME = False
if RESUME:
    from ultralytics import YOLO
    YOLO(run["last"]).train(resume=True)

## Summary and next step

- `runs/detect/<run>/weights/best.pt` is the trained model; hyperparameters and duration are saved in `reports/part3_training_*.json`.
- Not done here on purpose: the **test set** is not touched. It is used only once, in the evaluation notebook.

Next: `part3_04_evaluation.ipynb` (test-set metrics, confusion matrix, false negatives, IoU), then MLflow runs comparing `yolov8s` and `yolov8n`.